# Learning Phrase Representations using RNN Ecoder-Decoder for Statistical Machine Translation

In this second notebook on sequence-to-sequence models using PyTorch, we'll be implementing the model from [Learning Phrase Representations using RNN Encoder-Decoder for Statistical Machine Translation](https://arxiv.org/abs/1406.1078). This model will achieve improved test perplexity whilst only using a single layer RNN in both the encoder and the decoder.


## Introduction

Let's remind ourselves of the general encoder-decoder model. In the previous model, we used an multi-layered LSTM as the encoder and decoder.  
One downside of the previous model is that the decoder is trying to cram lots of information into the hidden states. Whilst decoding, the hidden state will need to contain information about the whole of the source sequence, as well as all of the tokens have been decoded so far. By alleviating some of this information compression, we can create a better model!

We'll also be using a GRU (Gated Recurrent Unit) instead of an LSTM (Long Short-Term Memory). Why? Mainly because that's what they did in the paper (this paper also introduced GRUs) and also because we used LSTMs last time. To understand how GRUs (and LSTMs) differ from standard RNNS, check out [this](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) link. Is a GRU better than an LSTM? [Research](https://arxiv.org/abs/1412.3555) has shown they're pretty much the same, and both are better than standard RNNs. 

## Preparing Data

All of the data preparation will be (almost) the same as last time, so we'll very briefly detail what each code block does. See the previous notebook for a recap.

We'll import PyTorch, TorchText, spaCy and a few standard modules.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
import spacy
import datasets
import tqdm
import evaluate
from collections import Counter

/opt/miniconda3/envs/nlp_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/miniconda3/envs/nlp_env/lib/python3.11/site-packages/torchvision/io/image.py:14: UserWarning: Failed to load image Python extension: 'dlopen(/opt/miniconda3/envs/nlp_env/lib/python3.11/site-packages/torchvision/image.so, 0x0006): Library not loaded: @rpath/libjpeg.9.dylib
  Referenced from: <EB3FF92A-5EB1-3EE8-AF8B-5923C1265422> /opt/miniconda3/envs/nlp_env/lib/python3.11/site-packages/torchvision/image.so
  Reason: tried: '/opt/miniconda3/envs/nlp_env/lib/python3.11/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/opt/miniconda3/envs/nlp_env/lib/python3.11/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/opt/miniconda3/envs/nlp_env/lib/python3.11/lib-dynload/../../libjpeg.9.dylib'

Then set a random seed for deterministic results/reproducability.

In [2]:
seed = 1234

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

### Dataset

We load the Multi30k dataset — ~30,000 parallel English-German image captions.

In [3]:
dataset = datasets.load_dataset("bentrevett/multi30k")

In [4]:
train_data, valid_data, test_data = (
    dataset["train"],
    dataset["validation"],
    dataset["test"],
)

### Tokenizers

We use spaCy to split sentences into tokens. Load the English and German models first:

In [5]:
en_nlp = spacy.load("en_core_web_sm")
de_nlp = spacy.load("de_core_news_sm")

Tokenize all examples: lowercase, trim to max length, and wrap with <sos> / <eos> tokens.

In [6]:
def tokenize_example(example, en_nlp, de_nlp, max_length, lowercase, sos_token, eos_token):
    en_tokens = [token.text for token in en_nlp.tokenizer(example["en"])][:max_length]
    de_tokens = [token.text for token in de_nlp.tokenizer(example["de"])][:max_length]

    if lowercase:
        en_tokens = [token.lower() for token in en_tokens]
        de_tokens = [token.lower() for token in de_tokens]

    en_tokens = [sos_token] + en_tokens + [eos_token]
    de_tokens = [sos_token] + de_tokens + [eos_token]

    return {"en_tokens": en_tokens, "de_tokens": de_tokens}

In [7]:
max_length = 1_000
lowercase = True
sos_token = "<sos>"
eos_token = "<eos>"

fn_kwargs = {
    "en_nlp": en_nlp,
    "de_nlp": de_nlp,
    "max_length": max_length,
    "lowercase": lowercase,
    "sos_token": sos_token,
    "eos_token": eos_token,
}

train_data = train_data.map(tokenize_example, fn_kwargs=fn_kwargs)
valid_data = valid_data.map(tokenize_example, fn_kwargs=fn_kwargs)
test_data = test_data.map(tokenize_example, fn_kwargs = fn_kwargs)

In [8]:
train_data[0]["en_tokens"]

['<sos>',
 'two',
 'young',
 ',',
 'white',
 'males',
 'are',
 'outside',
 'near',
 'many',
 'bushes',
 '.',
 '<eos>']

### Vocabularies

Build a vocabulary from training tokens. Tokens appearing fewer than `min_freq` times are treated as `<unk>`. Special tokens (`<unk>`, `<pad>`, `<sos>`, `<eos>`) are placed at indices 0-3.

In [9]:
class Vocab:
    def __init__(self, tokens_iterator, min_freq=1, specials=None):
        specials = specials or []
        counter = Counter()
        for tokens in tokens_iterator:
            counter.update(tokens)

        self.itos = list(specials) + [
            tok for tok, freq in counter.most_common()
            if freq >= min_freq and tok not in specials
        ]
        self.stoi = {tok: i for i, tok in enumerate(self.itos)}
        self.default_index = self.stoi[unk_token]

    def __getitem__(self, token):
        return self.stoi.get(token, self.default_index)

    def __contains__(self, token):
        return token in self.stoi

    def __len__(self):
        return len(self.itos)

    def set_default_index(self, index):
        self.default_index = index

    def lookup_tokens(self, indices):
        return [self.itos[i] for i in indices]

    def lookup_indices(self, tokens):
        return [self.stoi.get(token, self.default_index) for token in tokens]

In [10]:
min_freq = 2
unk_token = "<unk>"
pad_token = "<pad>"

special_tokens = [ unk_token, pad_token, sos_token, eos_token]

en_vocab = Vocab(train_data["en_tokens"], min_freq=min_freq, specials=special_tokens)
de_vocab = Vocab(train_data["de_tokens"], min_freq=min_freq, specials=special_tokens)

len(en_vocab), len(de_vocab)

(5893, 7853)

Check that both our vocabularies have the same index for the unknown and padding tokens as this simplifies some code later on.  
We also save the index of our `<unk>` and `<pad>` token, as we'll use it later

In [11]:
assert en_vocab[unk_token] == de_vocab[unk_token]
assert en_vocab[pad_token] == de_vocab[pad_token]

unk_index = en_vocab[unk_token]
pad_index = en_vocab[pad_token]

unk_index, pad_index

(0, 1)

In [12]:
en_vocab.set_default_index(unk_index)
de_vocab.set_default_index(unk_index)

### Numericalize

Neural networks work with numbers, not strings. We convert each token to its corresponding index using the vocabularies we just built.

In [13]:
def numericalize_example(example, en_vocab, de_vocab):
    en_ids = en_vocab.lookup_indices(example["en_tokens"])
    de_ids = de_vocab.lookup_indices(example["de_tokens"])

    return {"en_ids": en_ids, "de_ids": de_ids}

In [14]:
fn_kwargs = {"en_vocab": en_vocab, "de_vocab": de_vocab}

train_data = train_data.map(numericalize_example, fn_kwargs=fn_kwargs)
valid_data = valid_data.map(numericalize_example, fn_kwargs=fn_kwargs)
test_data = test_data.map(numericalize_example, fn_kwargs=fn_kwargs)

Each example now has `en_ids` and `de_ids` — lists of integers. We convert them to PyTorch tensors using `.with_format("torch")` so they're ready for the model.

In [15]:
data_type = "torch"
format_columns = ["en_ids", "de_ids"]

train_data = train_data.with_format(
    type=data_type, columns=format_columns, output_all_columns=True
)

valid_data = valid_data.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

test_data = test_data.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

In [16]:
train_data[0]["en_ids"], train_data[0]["de_ids"], type(train_data[0]["en_ids"])

(tensor([   2,   16,   24,   15,   25,  774,   17,   57,   80,  202, 1305,    5,
            3]),
 tensor([   2,   18,   26,  253,   30,   84,   20,   88,    7,   15,  110, 5374,
         3099,    4,    3]),
 torch.Tensor)

### Data Loaders

Sentences have different lengths, but PyTorch batches require tensors of the same shape. We solve this by padding shorter sequences with `<pad>` tokens to match the longest sequence in each batch.

The `collate_fn` handles this padding, and the `DataLoader` wraps our dataset into iterable batches.

In [17]:
def get_collate_fn(pad_index):
    def collate_fn(batch):
        batch_en_ids = [example["en_ids"] for example in batch]
        batch_de_ids = [example["de_ids"] for example in batch]

        batch_en_ids = nn.utils.rnn.pad_sequence(batch_en_ids, padding_value=pad_index)
        batch_de_ids = nn.utils.rnn.pad_sequence(batch_de_ids, padding_value=pad_index)

        return {"en_ids": batch_en_ids, "de_ids": batch_de_ids}
    return collate_fn

In [18]:
def get_data_loader(dataset, batch_size, pad_index, shuffle=False):
    # using the closures to create a collate fn with pad_index parameter
    collate_fn_pad = get_collate_fn(pad_index)
    
    data_loader = torch.utils.data.DataLoader(
        dataset = dataset,
        batch_size = batch_size,
        collate_fn = collate_fn_pad,
        shuffle = shuffle
    )

    return data_loader

In [19]:
batch_size = 128

train_data_loader = get_data_loader(train_data, batch_size, pad_index, shuffle=True)
valid_data_loader = get_data_loader(valid_data, batch_size, pad_index)
test_data_loader = get_data_loader(test_data, batch_size, pad_index)

## Building the Seq2Seq Model

We build three components: an **Encoder**, a **Decoder**, and a **Seq2Seq** wrapper that connects them.

### Encoder

The encoder is similar to the previous one, with the multi-layer LSTM swapped for a single-layer GRU. We also don't pass the dropout as an argument to the GRU as that dropout is used between each layer of a multi-layered RNN. As we only have a single layer, PyTorch will display a warning if we try and use pass a dropout value to it.

Another thing to note about the GRU is that it only requires and returns a hidden state, there is no cell state like in the LSTM.

$$\begin{align*}
h_t &= \text{GRU}(e(x_t), h_{t-1})\\
(h_t, c_t) &= \text{LSTM}(e(x_t), h_{t-1}, c_{t-1})\\
h_t &= \text{RNN}(e(x_t), h_{t-1})
\end{align*}$$

From the equations above, it looks like the RNN and the GRU are identical. Inside the GRU, however, is a number of *gating mechanisms* that control the information flow in to and out of the hidden state (similar to an LSTM). Again, for more info, check out [this](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) excellent post. 

The rest of the encoder should be very familar from the last tutorial, it takes in a sequence, $X = \{x_1, x_2, ... , x_T\}$, passes it through the embedding layer, recurrently calculates hidden states, $H = \{h_1, h_2, ..., h_T\}$, and returns a context vector (the final hidden state), $z=h_T$.

$$h_t = \text{EncoderGRU}(e(x_t), h_{t-1})$$

This is identical to the encoder of the general seq2seq model, with all the "magic" happening inside the GRU (green).

![](assets/seq2seq5.png)

In [20]:

class Encoder(nn.Module):
    def __init__(self, input_dim, embedding_dim, hidden_dim, dropout):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(input_dim, embedding_dim)
        self.rnn = nn.GRU(embedding_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, hidden = self.rnn(embedded)
        return hidden
        

### Decoder

The decoder is where the implementation differs significantly from the previous model and we alleviate some of the information compression.

Instead of the GRU in the decoder taking just the embedded target token, $d(y_t)$ and the previous hidden state $s_{t-1}$ as inputs, it also takes the context vector $z$. 

$$s_t = \text{DecoderGRU}(d(y_t), s_{t-1}, z)$$

Note how this context vector, $z$, does not have a $t$ subscript, meaning we re-use the same context vector returned by the encoder for every time-step in the decoder. 

Before, we predicted the next token, $\hat{y}_{t+1}$, with the linear layer, $f$, only using the top-layer decoder hidden state at that time-step, $s_t$, as $\hat{y}_{t+1}=f(s_t^L)$. Now, we also pass the embedding of current token, $d(y_t)$ and the context vector, $z$ to the linear layer.

$$\hat{y}_{t+1} = f(d(y_t), s_t, z)$$

Thus, our decoder now looks something like this:

![](assets/seq2seq6.png)

Note, the initial hidden state, $s_0$, is still the context vector, $z$, so when generating the first token we are actually inputting two identical context vectors into the GRU.

How do these two changes reduce the information compression? Well, hypothetically the decoder hidden states, $s_t$, no longer need to contain information about the source sequence as it is always available as an input. Thus, it only needs to contain information about what tokens it has generated so far. The addition of $y_t$ to the linear layer also means this layer can directly see what the token is, without having to get this information from the hidden state. 

However, this hypothesis is just a hypothesis, it is impossible to determine how the model actually uses the information provided to it (don't listen to anyone that says differently). Nevertheless, it is a solid intuition and the results seem to indicate that this modifications are a good idea!

Within the implementation, we will pass $d(y_t)$ and $z$ to the GRU by concatenating them together, so the input dimensions to the GRU are now `emb_dim + hid_dim` (as context vector will be of size `hid_dim`). The linear layer will take $d(y_t), s_t$ and $z$ also by concatenating them together, hence the input dimensions are now `emb_dim + hid_dim*2`. We also don't pass a value of dropout to the GRU as it only uses a single layer.

`forward` now takes a `context` argument. Inside of `forward`, we concatenate $y_t$ and $z$ as `emb_con` before feeding to the GRU, and we concatenate $d(y_t)$, $s_t$ and $z$ together as `output` before feeding it through the linear layer to receive our predictions, $\hat{y}_{t+1}$.

In [21]:
class Decoder(nn.Module):
    def __init__(self, output_dim, embedding_dim, hidden_dim, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(output_dim, embedding_dim)
        self.rnn = nn.GRU(embedding_dim + hidden_dim, hidden_dim)
        self.fc_out = nn.Linear(embedding_dim + hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, context):
        input = input.unsqueeze(0)
        # input = [1, batch_size]
        embedded = self.dropout(self.embedding(input))
        # embedded = [1, batch_size, embedding_dim]
        
        # hidden = [n layers * n directions, batch_size, hidden_dim]
        # context = [n layers * n directions, batch_size, hidden_dim]
        # n_layers and n_directions in the decoder will both always be 1, therefore:
        # hidden = [1, batch_size, hidden_dim]
        # context = [1, batch_size, hidden_dim]
        
        emb_con = torch.cat((embedded, context), dim=-1)        #  emb_con = [1, batch_size, embedding_dim + hidden_dim]
        output, hidden = self.rnn(emb_con, hidden)              
        # output = [seq len, batch_size, hidden_dim * n directions]
        # seq_len, n_layers and n_directions will always be 1 in this decoder, therefore:
        # output = [1, batch_size, hidden_dim]
        # hidden = [1, batch_size, hidden_dim]
        
        output = torch.cat((embedded.squeeze(0), hidden.squeeze(0), context.squeeze(0)), dim=1)
        # output = [batch_size, embedding_dim + hidden_dim * 2]
        prediction = self.fc_out(output)
        # prediction = [batch_size, output_dim]
        
        return prediction, hidden

### Seq2Seq Model

Putting the encoder and decoder together, we get:

![](assets/seq2seq7.png)

Again, in this implementation we need to ensure the hidden dimensions in both the encoder and the decoder are the same.

Briefly going over all of the steps:
- the `outputs` tensor is created to hold all predictions, $\hat{Y}$
- the source sequence, $X$, is fed into the encoder to receive a `context` vector
- the initial decoder hidden state is set to be the `context` vector, $s_0 = z = h_T$
- we use a batch of `<sos>` tokens as the first `input`, $y_1$
- we then decode within a loop:
  - inserting the input token $y_t$, previous hidden state, $s_{t-1}$, and the context vector, $z$, into the decoder
  - receiving a prediction, $\hat{y}_{t+1}$, and a new hidden state, $s_t$
  - we then decide if we are going to teacher force or not, setting the next input as appropriate (either the ground truth next token in the target sequence or the highest predicted next token)

In [22]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        assert (
            encoder.hidden_dim == decoder.hidden_dim
        ), "Hidden dimensions of encoder and decoder must be equal!"

    def forward(self, src, trg, teacher_forcing_ratio):
        trg_length, batch_size = trg.shape
        
        trg_vocab_size = self.decoder.output_dim
        outputs = torch.zeros(trg_length, batch_size, trg_vocab_size).to(self.device)

        # context is the last hidden state of the encoder
        context = self.encoder(src)
        # context also used as the initial hidden state of the decoder
        hidden = context

        # first input to the decoder is the <sos> tokens
        input = trg[0, :]

        for t in range(1, trg_length):
            # insert input token embedding, previous hidden state and the context state
            # receive output tensor (predictions) and new hidden state            
            output, hidden = self.decoder(input, hidden, context)
            # output = [batch size, output dim]
            # hidden = [1, batch size, hidden dim]
            
            # place predictions in a tensor holding predictions for each token
            outputs[t] = output

            # decide if we are going to use teacher forcing or not
            teacher_force = random.random() < teacher_forcing_ratio
            
            # get the highest predicted token from our predictions
            top1 = output.argmax(1)
            
            # if teacher forcing, use actual next token as next input
            # else, use the predicted token
            input = trg[t] if teacher_force else top1
            # input = [batch size]

        return outputs
        

## Training the Seq2Seq Model

### Model Initialization
We initialise our encoder, decoder and seq2seq model (placing it on the GPU if we have one). As before, the embedding dimensions and the amount of dropout used can be different between the encoder and the decoder, but the hidden dimensions must remain the same.

In [23]:
input_dim = len(de_vocab)
output_dim = len(en_vocab)
encoder_embedding_dim = 256
decoder_embedding_dim = 256
hidden_dim = 512
encoder_dropout = 0.5
decoder_dropout = 0.5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = Encoder(input_dim, encoder_embedding_dim, hidden_dim, encoder_dropout)
decoder = Decoder(output_dim, decoder_embedding_dim, hidden_dim, decoder_dropout)

model = Seq2Seq(encoder, decoder, device).to(device)

### Weights Initialization
Next, we initialize our parameters. The paper states the parameters are initialized from a normal distribution with a mean of 0 and a standard deviation of 0.01, i.e. $\mathcal{N}(0, 0.01)$. 

It also states we should initialize the recurrent parameters to a special initialization, however to keep things simple we'll also initialize them to $\mathcal{N}(0, 0.01)$.

In [24]:
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.normal_(param.data, mean=0, std=0.01)

model.apply(init_weights)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(7853, 256)
    (rnn): GRU(256, 512)
    (dropout): Dropout(p=0.5, inplace=False)
  )
  (decoder): Decoder(
    (embedding): Embedding(5893, 256)
    (rnn): GRU(768, 512)
    (fc_out): Linear(in_features=1280, out_features=5893, bias=True)
    (dropout): Dropout(p=0.5, inplace=False)
  )
)

In [25]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"The model has {count_parameters(model):,} trainable parameters")

The model has 14,219,781 trainable parameters


We initiaize our optimizer next...

In [26]:
optimizer = optim.Adam(model.parameters())

We also initialize the loss function, making sure to ignore the loss on `<pad>` tokens.

In [27]:
criterion = nn.CrossEntropyLoss(ignore_index=pad_index)

### Train and Eval funtions

We create the training loop...

In [28]:

def train_fn(model, data_loader, optimizer, criterion, clip, teacher_forcing_ratio, device):
    model.train()
    epoch_loss = 0
    for i, batch in enumerate(data_loader):
        # load src and trg using data_loader
        src = batch["de_ids"].to(device)                 # src = [src_length, batch_size]
        trg = batch["en_ids"].to(device)                 # trg = [trg_length, batch_size]
        
        optimizer.zero_grad()
        
        output = model(src, trg, teacher_forcing_ratio)  # output = [trg_length, batch_size, trg_vocab_size]

        # flattening the first two dims after removing <sos> from src and trg
        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)         # output = [trg_len-1 * batch_size, trg_vocab_size]
        trg = trg[1:].view(-1)                           #    trg = [trg_len-1 * batch_size]
        
        loss = criterion(output, trg)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
        
    return epoch_loss / len(data_loader)
    

...and the evaluation loop, remembering to set the model to `eval` mode and turn off teaching forcing.

In [29]:

def evaluate_fn(model, data_loader, criterion, device):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            src = batch["de_ids"].to(device)
            trg = batch["en_ids"].to(device)
    
            output = model(src, trg, teacher_forcing_ratio=0)
    
            output = output[1:].view(-1, output.shape[-1])
            trg = trg[1:].view(-1)
            loss = criterion(output, trg)
    
            epoch_loss += loss.item()
    return epoch_loss / len(data_loader)


### Model Training

Then, we train our model, saving the parameters that give us the best validation loss.

In [30]:
n_epochs = 10
clip = 1.0
teacher_forcing_ratio = 0.5

best_valid_loss = float("inf")

for epoch in tqdm.tqdm(range(n_epochs)):
    train_loss = train_fn(model, train_data_loader, optimizer, criterion, clip, teacher_forcing_ratio, device)
    valid_loss = evaluate_fn(model, valid_data_loader, criterion, device)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), "tut2-model.pt")

    print(f"\tTrain Loss: {train_loss:7.3f} | Train PPL: {np.exp(train_loss):7.3f}")
    print(f"\tValid Loss: {valid_loss:7.3f} | Valid PPL: {np.exp(valid_loss):7.3f}")

 10%|█████▎                                               | 1/10 [03:12<28:54, 192.75s/it]

	Train Loss:   5.031 | Train PPL: 153.107
	Valid Loss:   5.083 | Valid PPL: 161.323


 20%|██████████▌                                          | 2/10 [06:25<25:44, 193.03s/it]

	Train Loss:   4.379 | Train PPL:  79.723
	Valid Loss:   4.792 | Valid PPL: 120.598


 30%|███████████████▉                                     | 3/10 [09:35<22:20, 191.47s/it]

...

	Train Loss:   2.768 | Train PPL:  15.929
	Valid Loss:   3.844 | Valid PPL:  46.721


 90%|███████████████████████████████████████████████▋     | 9/10 [28:53<03:12, 192.82s/it]

	Train Loss:   2.551 | Train PPL:  12.814
	Valid Loss:   3.839 | Valid PPL:  46.485


100%|████████████████████████████████████████████████████| 10/10 [32:12<00:00, 193.30s/it]

	Train Loss:   2.370 | Train PPL:  10.696
	Valid Loss:   3.791 | Valid PPL:  44.320


## Evaluating the Model

The first thing to do is to test the model's performance on the test set.

We'll load the parameters (`state_dict`) that gave our model the best validation loss and run it on the test set to get our test loss and perplexity.


In [31]:

model.load_state_dict(torch.load("tut2-model.pt"))

test_loss = evaluate_fn(model, test_data_loader, criterion, device)

print(f"| Test Loss: {test_loss:.3f} | Test PPL: {np.exp(test_loss):7.3f} |")

/var/folders/58/kdxpbpcs4p368rfgpx57vt5m0000gn/T/ipykernel_74256/1084081046.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("tut2-model.

| Test Loss: 3.721 | Test PPL:  41.291 |


In [32]:

def translate_sentence(sentence, model, en_nlp, de_nlp, en_vocab, de_vocab, lowercase, sos_token, eos_token, device, max_output_length=25):
    model.eval()
    with torch.no_grad():
        if isinstance(sentence, str):
            tokens = [token.text for token in de_nlp.tokenizer(sentence)]
        else:
            tokens = [token for token in sentence]
    
        if lowercase:
            tokens = [token.lower() for token in tokens]
    
        tokens = [sos_token] + tokens + [eos_token]
        ids = de_vocab.lookup_indices(tokens)
        
        # 4. Create a batch with one sentence
        src_tensor = torch.LongTensor(ids).unsqueeze(-1).to(device)
    
        context = model.encoder(src_tensor)
        hidden = context
        inputs = en_vocab.lookup_indices([sos_token])
    
        for _ in range(max_output_length):
            inputs_tensor = torch.LongTensor([inputs[-1]]).to(device)
            output, hidden = model.decoder(inputs_tensor, hidden, context)
            predicted_token = output.argmax(-1).item()
            inputs.append(predicted_token)
            if predicted_token == en_vocab[eos_token]:
                break
        tokens = en_vocab.lookup_tokens(inputs)
    return tokens

In [33]:
sentence = test_data[0]["de"]
expected_translation = test_data[0]["en"]

sentence, expected_translation

('Ein Mann mit einem orangefarbenen Hut, der etwas anstarrt.',
 'A man in an orange hat starring at something.')

In [34]:
translation = translate_sentence(sentence, model, en_nlp, de_nlp, en_vocab, de_vocab, lowercase, sos_token, eos_token, device)

print(translation)

['<sos>', 'a', 'man', 'in', 'a', 'orange', 'jacket', 'is', 'working', 'with', 'a', '<unk>', '.', '<eos>']


In [35]:
sentence = "Ein Mann sieht sich einen Film an."
translation = translate_sentence(sentence, model, en_nlp, de_nlp, en_vocab, de_vocab, lowercase, sos_token, eos_token, device)
print(translation)

['<sos>', 'a', 'man', 'is', 'watching', 'a', 'a', '.', '<eos>']


In [36]:
translations = [
    translate_sentence(example["de"], model, en_nlp, de_nlp, en_vocab, de_vocab, lowercase, sos_token, eos_token, device)
    for example in tqdm.tqdm(test_data)
]

100%|█████████████████████████████████████████████████| 1000/1000 [00:12<00:00, 82.53it/s]


In [45]:
bleu = evaluate.load("bleu")

In [46]:
predictions = [" ".join(translation[1:-1]) for translation in translations]

references = [[example["en"]] for example in test_data]

In [47]:
def get_tokenizer_fn(nlp, lowercase):
    def tokenizer_fn(s):
        tokens = [token.text for token in nlp.tokenizer(s)]
        if lowercase:
            tokens = [token.lower() for token in tokens]
        return tokens
    return tokenizer_fn

In [48]:
tokenizer_fn = get_tokenizer_fn(en_nlp, lowercase)

In [49]:
results = bleu.compute(
    predictions=predictions, references=references, tokenizer=tokenizer_fn
)

In [50]:
results

{'bleu': 0.16890058809393158,
 'precisions': [0.5026047398928755,
  0.23327262649457597,
  0.11600309570900336,
  0.05983629692351115],
 'brevity_penalty': 1.0,
 'length_ratio': 1.043727982845765,
 'translation_length': 13629,
 'reference_length': 13058}

We got better performance/bleu score than the previous model. This is a pretty good sign that this model architecture is doing something right! Relieving the information compression seems like the way forward, and in the next part we'll expand on this even further with **attention**.